# LLM Zoomcamp 2026 – Homework 5: Monitoring

This notebook implements the Homework 5 tasks using OpenTelemetry and SQLite.

**Required files in the same folder:**
- `starter.py`
- `rag_helper.py`
- `.env` containing `OPENAI_API_KEY=...`


## 0. Install dependencies in the active notebook kernel

In [2]:
%pip install -q opentelemetry-api opentelemetry-sdk pandas python-dotenv openai minsearch gitsource
print("Dependencies installed. Restart the kernel once if imports still fail.")

/Users/veneraheddergott/LLM_zoomcamp/llm-zoomcamp_2026_vh-1/05-monitoring/llm-zoomcamp-hw5/.venv/bin/python3: No module named pip
Note: you may need to restart the kernel to use updated packages.
Dependencies installed. Restart the kernel once if imports still fail.


## 1. Check Python environment and files

In [2]:
import os
import sys
from pathlib import Path

print("Python:", sys.executable)
print("Working directory:", Path.cwd())
print("starter.py exists:", Path("starter.py").exists())
print("rag_helper.py exists:", Path("rag_helper.py").exists())
print(".env exists:", Path(".env").exists())

Python: /Users/veneraheddergott/LLM_zoomcamp/llm-zoomcamp_2026_vh-1/.venv/bin/python
Working directory: /Users/veneraheddergott/Downloads
starter.py exists: False
rag_helper.py exists: True
.env exists: False


In [3]:
from dotenv import load_dotenv

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is missing in .env"
print("OPENAI_API_KEY loaded successfully.")

OPENAI_API_KEY loaded successfully.


## 2. OpenTelemetry console exporter

In [4]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

# A provider can only be registered globally once per kernel.
# Restart the kernel before re-running this entire notebook from the top.
provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(ConsoleSpanExporter()))
trace.set_tracer_provider(provider)
tracer = trace.get_tracer("llm-zoomcamp")

print("OpenTelemetry console exporter configured.")

OpenTelemetry console exporter configured.


## 3. Load the starter RAG

In [6]:
from pathlib import Path

print("Aktueller Ordner:", Path.cwd())
print("Dateien:", [p.name for p in Path.cwd().iterdir()])

Aktueller Ordner: /Users/veneraheddergott/Downloads
Dateien: ['Tages-Planer.pdf', '216.pdf', 'notenbescheinigung20836De-13.pdf', '76b79c83-14a4-43f9-b3d3-e1842f4a8cbe-KI_Klausurfragen_2025.pdf', '978-3-658-48325-8.pdf', 's12559-023-10179-8.pdf', 'billing_history-2.pdf', 'OneDrive_1_2', 'Prävelenz_Fallzahlen.xlsx', '978-3-658-48987-8.pdf', 'Dashboard.pdf', 'IEEE Xplore Full-Text PDF:.pdf.pdf', 'Masterprojekt-17.pdf', 'WahlergebnisProjekte24o.pdf', 'exportExcel.xls', 'Ditzmann Transformation von Attributen.pptx', 'The-Impact-Potential-Assessment-Framework-IPAF-1.pdf', '.Rhistory', '978-3-658-35969-0.pdf', 'NA_Logo.pdf', 'WPM3820_22o-Abgabe Daten (Excel-Datei)-286144', 'Februar 2026 - 120455196.pdf', 'Webinar_KI_Sep_2023_short.pdf', 's10869-020-09707-2.pdf', 'international_declarations_angepasst.tex', 'ChatGPT Image 5. Juli 2026, 15_18_56.png', 'Agentenbasierte RAG-Systeme im Immobilienkontext_ Stand der Forschung, Architektur-Patterns und Fors.pdf', 's44336-024-00009-2.pdf', '978-3-658-

In [7]:
import inspect
import starter

rag = starter.rag
RAGBase = type(rag)

print("RAG class:", RAGBase)
print("RAG attributes:", sorted(vars(rag).keys()))
print("rag() signature:", inspect.signature(rag.rag))
print("search() signature:", inspect.signature(rag.search))
print("llm() signature:", inspect.signature(rag.llm))

ModuleNotFoundError: No module named 'starter'

## Q1. First trace

Create one span for each of these methods:
- `rag`
- `search`
- `llm`


In [ ]:
class RAGTraced(RAGBase):
    def search(self, *args, **kwargs):
        with tracer.start_as_current_span("search"):
            return super().search(*args, **kwargs)

    def llm(self, *args, **kwargs):
        with tracer.start_as_current_span("llm"):
            return super().llm(*args, **kwargs)

    def rag(self, *args, **kwargs):
        with tracer.start_as_current_span("rag"):
            return super().rag(*args, **kwargs)


# Reuse the already initialized starter object without guessing constructor arguments.
traced_rag = object.__new__(RAGTraced)
traced_rag.__dict__.update(rag.__dict__)

print("RAGTraced created successfully.")

In [ ]:
query = "How does the agentic loop keep calling the model until it stops?"

answer = traced_rag.rag(query)
print(answer)

print("\nQ1: Count the span entries named search, llm, and rag in the output above.")

### Q1 answer

Expected from this instrumentation: **3 spans**.

## Q2. Capture token usage and cost

In [ ]:
# Adjust these prices only if your selected model uses different rates.
# Values are USD per 1,000,000 tokens.
INPUT_PRICE_PER_MILLION = 0.0
OUTPUT_PRICE_PER_MILLION = 0.0


class RAGTracedWithMetrics(RAGBase):
    def search(self, *args, **kwargs):
        with tracer.start_as_current_span("search"):
            return super().search(*args, **kwargs)

    def llm(self, *args, **kwargs):
        with tracer.start_as_current_span("llm") as span:
            response = super().llm(*args, **kwargs)

            usage = getattr(response, "usage", None)
            if usage is not None:
                input_tokens = getattr(usage, "input_tokens", None)
                output_tokens = getattr(usage, "output_tokens", None)

                if input_tokens is not None:
                    span.set_attribute("input_tokens", input_tokens)
                if output_tokens is not None:
                    span.set_attribute("output_tokens", output_tokens)

                if input_tokens is not None and output_tokens is not None:
                    cost = (
                        input_tokens * INPUT_PRICE_PER_MILLION
                        + output_tokens * OUTPUT_PRICE_PER_MILLION
                    ) / 1_000_000
                    span.set_attribute("cost", cost)

            return response

    def rag(self, *args, **kwargs):
        with tracer.start_as_current_span("rag"):
            return super().rag(*args, **kwargs)


traced_rag_metrics = object.__new__(RAGTracedWithMetrics)
traced_rag_metrics.__dict__.update(rag.__dict__)

print("Metric-enabled RAG created.")

In [ ]:
answer = traced_rag_metrics.rag(query)
print(answer)

print("\nQ2: Read input_tokens from the llm span output and select the closest option.")

## Q3. Span timing

In the console output, compare `start_time` and `end_time` for the `llm` span. The later SQLite analysis will calculate durations automatically.

## Q4. Save spans to SQLite

In [ ]:
import sqlite3
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult


class SQLiteSpanExporter(SpanExporter):
    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path, check_same_thread=False)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self, timeout_millis=30000):
        return True


print("SQLiteSpanExporter defined.")

### Important

Restart the kernel now, then run the next setup cell instead of the earlier console-provider cell. OpenTelemetry allows only one global provider per kernel.

In [ ]:
# Run this cell after restarting the kernel and rerunning imports/class definitions as needed.
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor

sqlite_provider = TracerProvider()
sqlite_provider.add_span_processor(
    SimpleSpanProcessor(SQLiteSpanExporter("traces.db"))
)
trace.set_tracer_provider(sqlite_provider)
tracer = trace.get_tracer("llm-zoomcamp")

print("SQLite exporter configured.")

In [ ]:
# Recreate the metric-enabled object after the SQLite tracer is active.
traced_rag_metrics = object.__new__(RAGTracedWithMetrics)
traced_rag_metrics.__dict__.update(rag.__dict__)

answer = traced_rag_metrics.rag(query)
print(answer)

In [ ]:
import pandas as pd
import sqlite3

with sqlite3.connect("traces.db") as conn:
    spans_df = pd.read_sql_query("SELECT * FROM spans", conn)

display(spans_df)
print("Q4 span names:", sorted(spans_df["name"].dropna().unique().tolist()))

## Q5. Which child span takes the most total time?

In [ ]:
spans_df["duration_ms"] = (
    spans_df["end_time"] - spans_df["start_time"]
) / 1_000_000

duration_summary = (
    spans_df.loc[spans_df["name"] != "rag"]
    .groupby("name", as_index=False)["duration_ms"]
    .sum()
    .sort_values("duration_ms", ascending=False)
)

display(duration_summary)

if not duration_summary.empty:
    print("Q5 answer:", duration_summary.iloc[0]["name"])

## Q6. Token stability across four runs

In [ ]:
# The database already contains at least one call from Q4.
# Run the same query three additional times.
for run_number in range(1, 4):
    print(f"Run {run_number}/3")
    traced_rag_metrics.rag(query)

print("Three additional calls completed.")

In [ ]:
with sqlite3.connect("traces.db") as conn:
    llm_tokens = pd.read_sql_query(
        """
        SELECT rowid, input_tokens
        FROM spans
        WHERE name = 'llm' AND input_tokens IS NOT NULL
        ORDER BY rowid DESC
        LIMIT 4
        """,
        conn,
    ).sort_values("rowid")

display(llm_tokens)

tokens = llm_tokens["input_tokens"].astype(float)

if len(tokens) == 4:
    minimum = tokens.min()
    maximum = tokens.max()
    variation = 0.0 if minimum == 0 else (maximum - minimum) / minimum

    print(f"Minimum: {minimum:.0f}")
    print(f"Maximum: {maximum:.0f}")
    print(f"Variation: {variation:.2%}")

    if variation == 0:
        answer_q6 = "They're identical"
    elif variation <= 0.10:
        answer_q6 = "Within 10% of each other"
    elif variation <= 0.50:
        answer_q6 = "Within 50% of each other"
    else:
        answer_q6 = "They vary more than 50%"

    print("Q6 answer:", answer_q6)
else:
    print("Expected 4 llm rows, but found:", len(tokens))

## Final answers

Fill these values from your own outputs:

1. Number of spans: `...`
2. Input tokens: `...`
3. Typical LLM duration: `...`
4. Span names in SQLite: `...`
5. Slowest child span: `...`
6. Input-token variation: `...`
